In [ ]:
"""
priors_tic_v3.py
==================================================================
Pipeline para extraer priors (P_orb, P_rot, t0, duracion, profundidad
primaria/secundaria) de light curves TIC, y generar la figura de
diagnostico + catalogo priors.csv.

Es una fusion de priors_tic_v2.ipynb y priors_tic_optimizado_2_.ipynb,
con los siguientes arreglos motivados por el diagnostico que hicimos
sobre TIC 50897998 / sector 32:

1) DURACION DE BLS ADAPTATIVA
   v2 usaba un grid fijo de duraciones (0.01-0.08 d) que en datos reales
   quedo topado en el borde (mejor ajuste = 0.08 d exacto, el techo del
   grid). Aqui, si el mejor duration cae en el borde del grid, se
   ensancha automaticamente y se repite la busqueda (hasta un limite de
   intentos), y se avisa si aun asi no converge.

2) BUSQUEDA DE PERIODO EN DOS ETAPAS (coarse -> fine zoom)
   Una pasada gruesa sobre todo el rango (0.3-15 d por defecto) localiza
   el pico; una segunda pasada fina, acotada a +/-2% alrededor de ese
   pico, da mucha mas resolucion en periodo sin tener que barrer 10000+
   puntos en todo el rango.

3) MASCARA DE ECLIPSE CON SALVAGUARDA
   Con la duracion "real" (mas ancha) que aparece al arreglar (1), la
   mascara de eclipse (build_mask) puede llegar a cubrir el 100% de los
   datos, dejando el pipeline sin puntos para Lomb-Scargle (esto le paso
   a priors_tic_optimizado_2_ con este mismo archivo). Aqui, si la
   mascara cubre mas de max_mask_fraction del ciclo, se reduce el
   width_factor automaticamente antes de rendirse y usar la curva
   completa.

4) PROFUNDIDAD PRIMARIA Y SECUNDARIA por separado (idea de optimizado),
   cada una con su error, y compatible con binarias eclipsantes.

5) load_lightcurve robusto (idea de optimizado): detecta separador,
   valida columnas, ordena por tiempo, quita duplicados. En el archivo
   de prueba no cambiaba nada (ya venia limpio), pero protege contra
   archivos que no lo esten.

6) FIGURA DE DIAGNOSTICO combinada: scatter crudo + curva binneada
   (para que se vea "limpia" sin que eso disfrace un mal ajuste),
   periodograma BLS con el pico marcado, fase rotacional, y un panel de
   "salud" del ajuste (cobertura real de la mascara, si se activo el
   fallback, cuantas veces se tuvo que ensanchar el grid de duracion).

7) Catalogo priors.csv keyeado por (TIC, Sector), igual que optimizado.

Requiere: numpy, pandas, matplotlib, astropy, wotan.
"""

import os
import re
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from astropy.timeseries import BoxLeastSquares, LombScargle
from wotan import flatten


# ==================================================================
# CONFIGURACION POR DEFECTO
# ==================================================================

DEFAULTS = dict(
    time_column="#Time",
    flux_column="Flux_CBV_SS",

    # --- detrending (wotan) ---
    window_length=0.55,
    break_tolerance=0.5,

    # --- BLS: rango de periodo ---
    min_period_bls=0.3,
    max_period_bls=15.0,
    n_periods_coarse=3000,
    n_periods_fine=4000,
    fine_zoom_frac=0.02,       # +/- 2% del periodo alrededor del pico, en la pasada fina

    # --- BLS: grid de duracion (adaptativo) ---
    duration_min_frac=0.005,   # fraccion de min_period_bls
    duration_max_frac=0.12,    # fraccion de min_period_bls, punto de partida
    n_durations=40,
    duration_grow_factor=1.6,  # factor de ensanche si el mejor ajuste toca el borde
    max_duration_grid_tries=4,

    # --- mascara de eclipse (para aislar la rotacion) ---
    width_factor=3.0,
    max_mask_fraction=0.7,     # si la mascara cubre mas que esto, se reduce width_factor
    min_points_out_of_eclipse=30,

    # --- Lomb-Scargle (periodo de rotacion) ---
    min_period_ls=0.2,
    max_period_ls=15.0,
    n_freq_ls=5000,

    # --- profundidad de eclipse ---
    eclipse_half_width=0.03,
    out_of_eclipse_lo=0.15,
    out_of_eclipse_hi=0.35,
)


# ==================================================================
# 1) CARGA DEL LIGHT CURVE
# ==================================================================

def load_lightcurve(filename, time_column="#Time", flux_column="Flux_CBV_SS", sep=None):
    """Carga robusta: detecta separador, valida columnas, ordena por
    tiempo y quita timestamps duplicados (que rompen BLS/LS)."""
    if not os.path.exists(filename):
        raise FileNotFoundError(f"No se encontro el archivo: {filename}")

    if sep is None:
        sep = "," if filename.lower().endswith(".csv") else r"\s+"

    df = pd.read_csv(filename, sep=sep, engine="python")
    df.columns = [c.strip() for c in df.columns]

    if time_column not in df.columns or flux_column not in df.columns:
        raise KeyError(
            f"Columnas '{time_column}' / '{flux_column}' no encontradas.\n"
            f"Columnas disponibles: {list(df.columns)}"
        )

    time = df[time_column].values.astype(float)
    flux = df[flux_column].values.astype(float)

    mask = np.isfinite(time) & np.isfinite(flux)
    time, flux = time[mask], flux[mask]

    order = np.argsort(time)
    time, flux = time[order], flux[order]
    _, unique_idx = np.unique(time, return_index=True)
    time, flux = time[unique_idx], flux[unique_idx]

    if len(time) < 50:
        raise ValueError(f"Muy pocos puntos validos ({len(time)}) tras limpiar el archivo.")

    return time, flux


def parse_tic_sector_from_filename(filename):
    name = os.path.basename(filename)
    m = re.search(r"tic\s*0*(\d+).*?_S_0*(\d+)", name, re.IGNORECASE)
    if m:
        return int(m.group(1)), int(m.group(2))
    return None, None


# ==================================================================
# 2) BLS ADAPTATIVO + DOS ETAPAS
# ==================================================================

def _bls_once(time, flux, min_period, max_period, n_periods, min_dur_period, cfg):
    """Una corrida de BLS, ensanchando el grid de duracion si el mejor
    ajuste queda pegado al borde superior."""
    frac_max = cfg["duration_max_frac"]
    bls = BoxLeastSquares(time, flux)

    for attempt in range(1, cfg["max_duration_grid_tries"] + 1):
        durations = np.linspace(
            cfg["duration_min_frac"] * min_dur_period,
            frac_max * min_dur_period,
            cfg["n_durations"],
        )
        periods = np.linspace(min_period, max_period, n_periods)
        result = bls.power(periods, durations)
        idx = np.argmax(result.power)
        best_duration = result.duration[idx]

        hit_ceiling = np.isclose(best_duration, durations[-1], rtol=1e-6)
        if not hit_ceiling:
            break
        frac_max *= cfg["duration_grow_factor"]
    else:
        warnings.warn(
            f"BLS: la duracion optima ({best_duration:.4f} d) sigue en el techo del "
            f"grid tras {attempt} intentos ensanchando. Revisa manualmente: puede ser "
            "una senal de baja frecuencia, no un eclipse real."
        )

    return {
        "P_bls": result.period[idx],
        "t0": result.transit_time[idx],
        "duration": best_duration,
        "period_grid": result.period,
        "power_grid": result.power,
        "duration_grid_max": durations[-1],
        "grid_tries": attempt,
    }


def run_bls_two_stage(time, flux, cfg):
    """Pasada gruesa en todo el rango + pasada fina (zoom) alrededor del
    pico, para precision alta sin pagar el costo de un grid ultra denso
    en todo 0.3-15 d."""
    coarse = _bls_once(
        time, flux,
        cfg["min_period_bls"], cfg["max_period_bls"], cfg["n_periods_coarse"],
        min_dur_period=cfg["min_period_bls"], cfg=cfg,
    )

    P0 = coarse["P_bls"]
    lo = max(cfg["min_period_bls"], P0 * (1 - cfg["fine_zoom_frac"]))
    hi = min(cfg["max_period_bls"], P0 * (1 + cfg["fine_zoom_frac"]))

    fine = _bls_once(
        time, flux, lo, hi, cfg["n_periods_fine"],
        min_dur_period=cfg["min_period_bls"], cfg=cfg,
    )
    fine["coarse"] = coarse
    return fine


def determine_orbital_period(time, flux, t0, P_bls):
    """Decide si el periodo real es P_bls o 2*P_bls comparando la
    profundidad del minimo primario vs secundario."""
    P1, P2 = P_bls, 2 * P_bls
    phase = ((time - t0) / P2) % 1
    primary = np.nanmedian(flux[np.abs(phase - 0.0) < 0.05])
    secondary = np.nanmedian(flux[np.abs(phase - 0.5) < 0.05])
    if abs(primary - secondary) > 0.005:
        return P2, True
    return P1, False


# ==================================================================
# 3) MASCARA DE ECLIPSE CON SALVAGUARDA
# ==================================================================

def build_mask_safe(time, period, t0, duration, cfg):
    """Igual que build_mask original, pero si la mascara cubre mas de
    max_mask_fraction del ciclo, reduce width_factor automaticamente
    antes de rendirse."""
    width_factor = cfg["width_factor"]
    mask = np.zeros(len(time), dtype=bool)
    frac = 1.0

    for _ in range(6):
        phase = ((time - t0) / period) % 1
        width = min(width_factor * duration / period, 0.5)
        primary = (phase < width) | (phase > 1 - width)
        secondary = np.abs(phase - 0.5) < width
        mask = primary | secondary
        frac = mask.mean()
        if frac <= cfg["max_mask_fraction"]:
            break
        width_factor *= 0.7

    return mask, width_factor, frac


# ==================================================================
# 4) PROFUNDIDAD DE ECLIPSE (primario y secundario)
# ==================================================================

def compute_depth(time, flux, t0, period, half_width=0.03, out_lo=0.15, out_hi=0.35):
    phase = ((time - t0) % period) / period
    out_eclipse = (phase > out_lo) & (phase < out_hi)

    def _depth_at(center):
        in_ecl = np.abs(((phase - center + 0.5) % 1) - 0.5) < half_width
        if np.sum(in_ecl) < 5 or np.sum(out_eclipse) < 5:
            return None, None
        baseline = np.nanmedian(flux[out_eclipse])
        eclipse = np.nanmedian(flux[in_ecl])
        depth = baseline - eclipse
        depth_err = np.nanstd(flux[in_ecl]) / np.sqrt(np.sum(in_ecl))
        return float(depth), float(depth_err)

    return {"primary": _depth_at(0.0), "secondary": _depth_at(0.5)}


# ==================================================================
# 5) LOMB-SCARGLE (periodo de rotacion)
# ==================================================================

def run_ls(time, flux, min_period, max_period, n_freq):
    freq = np.linspace(1 / max_period, 1 / min_period, n_freq)
    ls = LombScargle(time, flux)
    power = ls.power(freq)
    idx = np.argmax(power)
    return {"P_rot": 1 / freq[idx], "frequency": freq, "power": power}


# ==================================================================
# 6) PIPELINE PRINCIPAL
# ==================================================================

def analyze_system(filename, **kwargs):
    cfg = {**DEFAULTS, **kwargs}

    time, flux = load_lightcurve(filename, cfg["time_column"], cfg["flux_column"])

    # -------- detrend --------
    orbital_flux, rotational_trend = flatten(
        time, flux,
        method="biweight",
        window_length=cfg["window_length"],
        break_tolerance=cfg["break_tolerance"],
        return_trend=True,
    )

    # -------- BLS (adaptativo, dos etapas) --------
    orbital = run_bls_two_stage(time, orbital_flux, cfg)
    P_orb, is_2P = determine_orbital_period(time, orbital_flux, orbital["t0"], orbital["P_bls"])

    # -------- profundidad --------
    depths = compute_depth(
        time, orbital_flux, orbital["t0"], P_orb,
        half_width=cfg["eclipse_half_width"],
        out_lo=cfg["out_of_eclipse_lo"], out_hi=cfg["out_of_eclipse_hi"],
    )

    # -------- mascara de eclipse (con salvaguarda) --------
    mask, width_factor_used, mask_fraction = build_mask_safe(
        time, P_orb, orbital["t0"], orbital["duration"], cfg
    )

    time_rot, flux_rot = time[~mask], flux[~mask]
    fallback_used = len(time_rot) < cfg["min_points_out_of_eclipse"]
    if fallback_used:
        warnings.warn(
            f"Solo quedaron {len(time_rot)} puntos fuera de eclipse "
            f"({mask_fraction*100:.1f}% enmascarado incluso tras reducir width_factor "
            f"a {width_factor_used:.2f}). Se usa la curva completa para Lomb-Scargle."
        )
        time_rot, flux_rot = time, flux

    rotation = run_ls(time_rot, flux_rot, cfg["min_period_ls"], cfg["max_period_ls"], cfg["n_freq_ls"])

    tic, sector = parse_tic_sector_from_filename(filename)

    return {
        "filename": filename, "tic": tic, "sector": sector,
        "time": time, "flux": flux,
        "orbital_flux": orbital_flux, "rotational_flux": rotational_trend,
        "time_rot": time_rot, "flux_rot": flux_rot, "mask": mask,

        "P_bls": orbital["P_bls"], "P_orb": P_orb, "is_2P": is_2P,
        "P_rot": rotation["P_rot"],
        "t0": orbital["t0"], "duration": orbital["duration"],

        "depth_primary": depths["primary"][0], "depth_primary_err": depths["primary"][1],
        "depth_secondary": depths["secondary"][0], "depth_secondary_err": depths["secondary"][1],

        "orbital": orbital, "rotation": rotation,

        # --- diagnostico de salud del ajuste ---
        "mask_fraction": mask_fraction,
        "width_factor_used": width_factor_used,
        "fallback_used": fallback_used,
        "duration_grid_tries": orbital["grid_tries"],
        "duration_hit_ceiling": np.isclose(orbital["duration"], orbital["duration_grid_max"], rtol=1e-6),

        "cfg": cfg,
    }


# ==================================================================
# 7) FIGURA DE DIAGNOSTICO
# ==================================================================

def plot_diagnostics(results, savepath=None, nbins_orb=100, nbins_rot=50):
    time, flux = results["time"], results["flux"]
    orbital_flux = results["orbital_flux"]
    rotational_flux = results["rotational_flux"]
    P_orb, P_rot, t0 = results["P_orb"], results["P_rot"], results["t0"]

    phase_orb = ((time - t0) / P_orb) % 1
    phase_rot = ((time - np.nanmin(time)) / P_rot) % 1
    idx_orb, idx_rot = np.argsort(phase_orb), np.argsort(phase_rot)

    def _bin(phase, y, nbins, min_count=5):
        bins = np.linspace(0, 1, nbins + 1)
        digit = np.digitize(phase, bins)
        pb, yb = [], []
        for k in range(1, len(bins)):
            m = digit == k
            if m.sum() > min_count:
                pb.append(np.nanmedian(phase[m]))
                yb.append(np.nanmedian(y[m]))
        return np.array(pb), np.array(yb)

    pb_orb, fb_orb = _bin(phase_orb, orbital_flux, nbins_orb)
    pb_rot, fb_rot = _bin(phase_rot, rotational_flux, nbins_rot)

    fig = plt.figure(figsize=(13, 11))
    gs = GridSpec(3, 2, height_ratios=[1, 1, 0.55], hspace=0.40, wspace=0.22)

    ax_a = fig.add_subplot(gs[0, 0])
    ax_b = fig.add_subplot(gs[0, 1])
    ax_c = fig.add_subplot(gs[1, 0])
    ax_d = fig.add_subplot(gs[1, 1])
    ax_e = fig.add_subplot(gs[2, :])

    # (a) detrending
    ax_a.plot(time, flux, ".", ms=1, alpha=0.25, color="#555555", label="Original")
    ax_a.plot(time, rotational_flux, lw=1.5, color="#ff7f0e", label="Tendencia (wotan)")
    ax_a.set_title("(a) Detrending"); ax_a.set_xlabel("Time (d)"); ax_a.set_ylabel("Flux")
    ax_a.legend(fontsize=8)

    # (b) periodograma BLS con pico marcado
    pg, powg = results["orbital"]["coarse"]["period_grid"], results["orbital"]["coarse"]["power_grid"]
    ax_b.plot(pg, powg, lw=0.7, color="#7b3294")
    ax_b.axvline(results["P_bls"], color="red", ls="--", lw=1, label=f"P_bls={results['P_bls']:.5f} d")
    ax_b.set_title("(b) Periodograma BLS (pasada gruesa)")
    ax_b.set_xlabel("Periodo (d)"); ax_b.set_ylabel("Power")
    ax_b.legend(fontsize=8)

    # (c) orbital: scatter crudo + binneado
    ax_c.plot(phase_orb[idx_orb], orbital_flux[idx_orb], ".", ms=2, alpha=0.30, color="#7b3294")
    ax_c.plot(pb_orb, fb_orb, "-", color="black", lw=1.8)
    title_c = f"(c) Curva aplanada, fase orbital\nP_orb={P_orb:.5f} d" + (" (=2xP_bls)" if results["is_2P"] else "")
    if results["depth_primary"] is not None:
        title_c += f"  |  prof.prim={results['depth_primary']*100:.3f}%"
    ax_c.set_title(title_c, fontsize=10)
    ax_c.set_xlabel("Phase"); ax_c.set_ylabel("Flux normalizado")

    # (d) rotacional
    ax_d.plot(phase_rot[idx_rot], rotational_flux[idx_rot], ".", ms=2, alpha=0.30, color="#1f77b4")
    ax_d.plot(pb_rot, fb_rot, "-", color="black", lw=1.8)
    ax_d.set_title(f"(d) Fase rotacional\nP_rot={P_rot:.5f} d", fontsize=10)
    ax_d.set_xlabel("Phase"); ax_d.set_ylabel("Flux")

    # (e) panel de salud del ajuste (texto)
    ax_e.axis("off")
    health_lines = [
        f"TIC {results['tic']}   Sector {results['sector']}   |   archivo: {os.path.basename(results['filename'])}",
        f"P_bls = {results['P_bls']:.6f} d   ->   P_orb adoptado = {P_orb:.6f} d "
        f"({'2 x P_bls' if results['is_2P'] else '1 x P_bls (sin evidencia de eclipse secundario distinto)'})",
        f"duracion BLS = {results['duration']:.5f} d   |   grid de duracion ensanchado {results['duration_grid_tries']} vez/veces"
        + ("   [!] SIGUE en el techo del grid, revisar a mano" if results["duration_hit_ceiling"] else "   [ok]"),
        f"profundidad primaria = {results['depth_primary']*100:.4f}% +/- {results['depth_primary_err']*100:.4f}%"
        if results["depth_primary"] is not None else "profundidad primaria = no medible",
        f"profundidad secundaria = {results['depth_secondary']*100:.4f}% +/- {results['depth_secondary_err']*100:.4f}%"
        if results["depth_secondary"] is not None else "profundidad secundaria = no medible",
        f"mascara de eclipse: {results['mask_fraction']*100:.1f}% de los puntos "
        f"(width_factor usado = {results['width_factor_used']:.2f}, original = {results['cfg']['width_factor']})"
        + ("   [!] fallback activado: se uso la curva completa para Lomb-Scargle" if results["fallback_used"] else "   [ok]"),
    ]
    ax_e.text(0.0, 1.0, "\n".join(health_lines), fontsize=10, family="monospace",
               va="top", ha="left", transform=ax_e.transAxes)

    plt.tight_layout()
    if savepath:
        plt.savefig(savepath, dpi=180, bbox_inches="tight")
    plt.show()


# ==================================================================
# 8) GUARDAR PRIORS EN EL CATALOGO
# ==================================================================

def save_priors(results, tic=None, sector=None, quality="A", notes="",
                 filename="priors.csv", overwrite=None):
    tic = tic if tic is not None else results.get("tic")
    sector = sector if sector is not None else results.get("sector")
    if tic is None:
        raise ValueError("No se pudo determinar el TIC: pasalo explicitamente.")

    row = {
        "TIC": tic, "Sector": sector,
        "P_bls": results["P_bls"], "P_orb": results["P_orb"], "is_2P": results["is_2P"],
        "P_rot": results["P_rot"], "t0": results["t0"], "Duration": results["duration"],
        "Depth_primary": results["depth_primary"], "Depth_primary_err": results["depth_primary_err"],
        "Depth_secondary": results["depth_secondary"], "Depth_secondary_err": results["depth_secondary_err"],
        "Mask_fraction": results["mask_fraction"], "Width_factor_used": results["width_factor_used"],
        "Fallback_used": results["fallback_used"], "Duration_grid_tries": results["duration_grid_tries"],
        "Duration_hit_ceiling": results["duration_hit_ceiling"],
        "Quality": quality, "Notes": notes,
    }

    if not os.path.exists(filename):
        pd.DataFrame([row]).to_csv(filename, index=False)
        print(f"Catalogo creado: {filename}")
        return

    df = pd.read_csv(filename)
    same_tic = df["TIC"] == tic
    same_sector = df["Sector"].isna() if sector is None else df["Sector"] == sector
    mask = same_tic & same_sector
    label = f"TIC {tic} / sector {sector}"

    if mask.any():
        do_overwrite = overwrite
        if do_overwrite is None:
            do_overwrite = not sys.stdin.isatty()
            if not sys.stdin.isatty():
                print(f"{label} ya existe; sobreescribiendo (modo no interactivo).")
        if do_overwrite:
            # en vez de asignar celda por celda (rompe con pandas moderno si una
            # columna quedo con dtype float por haber tenido "" -> NaN al releer
            # el csv), se quita la fila vieja y se agrega la nueva ya tipada bien
            df = pd.concat([df.loc[~mask], pd.DataFrame([row])], ignore_index=True)
            print(f"{label}: entrada actualizada.")
        else:
            print(f"{label}: ya existe, no se sobreescribe (overwrite=False).")
    else:
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
        print(f"{label}: fila nueva agregada.")

    df.to_csv(filename, index=False)
    print(f"Guardado en {filename}")


# ==================================================================
# 9) PIPELINE DE UN SOLO PASO
# ==================================================================

def run_pipeline(filename, tic=None, sector=None, quality="A", notes="",
                  show_plot=True, save_csv=True, catalog="priors.csv",
                  savepath=None, **kwargs):
    results = analyze_system(filename, **kwargs)
    if tic is not None:
        results["tic"] = tic
    if sector is not None:
        results["sector"] = sector

    print(f"Archivo:   {filename}")
    print(f"TIC:       {results['tic']}   Sector: {results['sector']}")
    print(f"P_BLS  =   {results['P_bls']:.6f} d")
    print(f"P_orb  =   {results['P_orb']:.6f} d  ({'2xP_bls' if results['is_2P'] else '1xP_bls'})")
    print(f"P_rot  =   {results['P_rot']:.6f} d")
    print(f"t0     =   {results['t0']:.6f}")
    print(f"duration = {results['duration']:.6f} d  (grid ensanchado {results['duration_grid_tries']} vez/veces)")
    if results["depth_primary"] is not None:
        print(f"depth (primario)   = {results['depth_primary']*100:.4f}% +/- {results['depth_primary_err']*100:.4f}%")
    if results["depth_secondary"] is not None:
        print(f"depth (secundario) = {results['depth_secondary']*100:.4f}% +/- {results['depth_secondary_err']*100:.4f}%")
    print(f"mascara de eclipse = {results['mask_fraction']*100:.1f}% "
          f"(width_factor usado={results['width_factor_used']:.2f}) "
          f"{'[fallback activado]' if results['fallback_used'] else ''}")

    if show_plot:
        plot_diagnostics(results, savepath=savepath)

    if save_csv:
        save_priors(results, tic=results["tic"], sector=results["sector"],
                    quality=quality, notes=notes, filename=catalog)

    return results


# ==================================================================
# USO
# ==================================================================
if __name__ == "__main__":
    file = "/home/luciam/Documents/starry/Orion_yceb/dat_orion_sergio/sample_3/831359(1)/Light_Curve_Files/tic 50897998_S_32_LC.txt"   # <-- cambia esto por tu archivo
    results = run_pipeline(
        file,
        window_length=0.55,
        show_plot=True,
        save_csv=True,
        catalog="priors.csv",
        quality="A",
        notes="",
        savepath="diagnostico_v3.png",
    )